In [1]:
import os
import sys

# 一つ上のディレクトリの絶対パスを取得して sys.path に追加
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

# クラスをロード
from Analysis import NeuMoAnalyzer

In [2]:
MULTIRUN_PATH = '../multirun/2026-06-25/17-03-44'
analyzer = NeuMoAnalyzer(MULTIRUN_PATH)

In [ ]:
# 2. Stage 1: スパイク情報の非同期・一括抽出（最も重いI/Oをここで分離）
# 2回目以降の解析では、この保存された pkl を直ロードすればOKです
df_spikes = analyzer.extract_and_save_spikes(output_dir="extracted_data")

# 3. Stage 2-A: 発火率レポートの生成（一瞬で終わります）
df_rate = analyzer.generate_firing_rate_report(
    df_spikes, output_dir="extracted_data"
)

# 4. Stage 2-B: 実効ランクレポートの生成（同一ファイルの重複ロードが排除され劇的に高速化）
df_rank = analyzer.generate_effective_rank_report(
    df_spikes, window_ms=5.0, output_dir="extracted_data"
)

=== Stage 1: Extracting Spike Indices ===


Jobs: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 160/160 [36:26<00:00, 13.66s/it]


Spike metadata saved to: extracted_data/df_spikes_meta_20260627_020639.pkl
=== Stage 2-A: Generating Firing Rate Report ===
Firing rate report saved to: extracted_data/df_rate_20260627_020640.csv
=== Stage 2-B: Generating Effective Rank Report ===


Processing Ranks by File:   1%|▉                                                                                                                          | 76/9600 [00:17<34:56,  4.54it/s]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if not df_rank.empty:
    # 軸の限界値を統一して比較しやすくする
    x_min, x_max = df_rank["er_orig"].min(), df_rank["er_orig"].max()
    y_min, y_max = df_rank["er_res"].min(), df_rank["er_res"].max()
    x_range = [x_min * 0.9, x_max * 1.1]
    y_range = [y_min * 0.9, y_max * 1.1]

    # 平均（mean）とシード（seed）の組み合わせを取得
    conditions = df_rank.groupby(["syn_loc_mean", "seed"]).size().index
    n_conds = len(conditions)

    fig, axes = plt.subplots(1, n_conds, figsize=(5 * n_conds, 5), squeeze=False)

    for i, (m, s_val) in enumerate(conditions):
        ax = axes[0, i]
        subset = df_rank[
            (df_rank["syn_loc_mean"] == m) & (df_rank["seed"] == s_val)
        ]

        plotted_any = False
        # dAP から dSpike への変更を反映
        for s_type, color, label in [
            ("bAP", "cyan", "bAP"),
            ("dSpike", "orange", "dSpike"),
        ]:
            data = subset[subset["spike_type"] == s_type]
            if not data.empty:
                ax.scatter(
                    data["er_orig"],
                    data["er_res"],
                    alpha=0.6,
                    c=color,
                    label=label,
                    edgecolors="k",
                    s=25,
                )
                plotted_any = True

        ax.set_xlim(x_range)
        ax.set_ylim(y_range)
        ax.set_title(f"Mean: {m}, Seed: {s_val}")
        ax.set_xlabel("ER (Original Ca)")
        if i == 0:
            ax.set_ylabel("ER (Residual Ca)")

        if plotted_any:
            ax.legend(loc="upper right")
        ax.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()
else:
    print("df_rank が空のためプロットできません。")

In [ ]:
from sklearn.linear_model import LinearRegression

results = []
conditions = df_rank.groupby(["syn_loc_mean", "syn_loc_std", "seed"]).size().index

# 各条件・スパイクタイプごとに線形回帰を実行
for m, s, seed_val in conditions:
    subset = df_rank[
        (df_rank["syn_loc_mean"] == m)
        & (df_rank["syn_loc_std"] == s)
        & (df_rank["seed"] == seed_val)
    ]

    for s_type in ["bAP", "dSpike"]:
        data = subset[subset["spike_type"] == s_type]

        if len(data) > 1:
            X = data[["er_orig"]].values
            y = data["er_res"].values

            model = LinearRegression()
            model.fit(X, y)

            results.append(
                {
                    "syn_loc_mean": m,
                    "syn_loc_std": s,
                    "seed": seed_val,
                    "spike_type": s_type,
                    "slope": model.coef_[0],
                    "intercept": model.intercept_,
                }
            )

res_df = pd.DataFrame(results)

if not res_df.empty:
    # 統計値の計算
    summary_df = (
        res_df.groupby(["syn_loc_mean", "spike_type"])
        .agg({"slope": ["mean", "std"], "intercept": ["mean", "std"]})
        .reset_index()
    )
    summary_df.columns = [
        f"{col[0]}_{col[1]}" if col[1] else col[0] for col in summary_df.columns
    ]

    # 2x2 プロットの描画
    fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True)
    types = ["bAP", "dSpike"]
    params = ["slope", "intercept"]
    titles = ["Slope (傾き)", "Intercept (切片)"]

    for i, p_name in enumerate(params):
        for j, t_name in enumerate(types):
            ax = axes[i, j]
            plot_data = summary_df[summary_df["spike_type"] == t_name]

            m_vals = plot_data["syn_loc_mean"]
            mean_vals = plot_data[f"{p_name}_mean"]
            std_vals = plot_data[f"{p_name}_std"].fillna(0)

            color = "cyan" if t_name == "bAP" else "orange"

            ax.plot(
                m_vals,
                mean_vals,
                marker="o",
                linestyle="-",
                color=color,
                label=f"{t_name}",
            )
            ax.fill_between(
                m_vals,
                mean_vals - std_vals,
                mean_vals + std_vals,
                color=color,
                alpha=0.15,
            )

            ax.set_title(f"{t_name} - {titles[i]}")
            ax.set_ylabel(titles[i])
            ax.grid(True, linestyle="--", alpha=0.5)

            if i == 1:
                ax.set_xlabel("syn_loc_mean")

    plt.tight_layout()
    plt.show()

In [ ]:
# 2. 窓幅を 1.0 ms に指定して、実効ランクを再計算（Stage 2-B の再実行）
# 同一ファイルのロードは1回に最適化されているため、高速に処理されます
print("窓幅を 1ms に変更して実効ランクを再計算中...")
df_rank_1ms = analyzer.generate_effective_rank_report(
    df_spikes,
    window_ms=1.0,  # 5.0ms から 1.0ms に変更
    output_dir="extracted_data",
)

# 3. データの確認
print("\n--- 計算完了 ---")
print(df_rank_1ms.head())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# ==============================================================================
# 1. 散布図の再描画（Original vs Residual @ 1.0ms window）
# ==============================================================================
if "df_rank_1ms" in locals() and not df_rank_1ms.empty:
    # 軸の限界値を統一
    x_min, x_max = df_rank_1ms["er_orig"].min(), df_rank_1ms["er_orig"].max()
    y_min, y_max = df_rank_1ms["er_res"].min(), df_rank_1ms["er_res"].max()
    x_range = [x_min * 0.9, x_max * 1.1]
    y_range = [y_min * 0.9, y_max * 1.1]

    conditions = df_rank_1ms.groupby(["syn_loc_mean", "seed"]).size().index
    n_conds = len(conditions)

    fig, axes = plt.subplots(1, n_conds, figsize=(5 * n_conds, 5), squeeze=False)

    for i, (m, s_val) in enumerate(conditions):
        ax = axes[0, i]
        subset = df_rank_1ms[
            (df_rank_1ms["syn_loc_mean"] == m) & (df_rank_1ms["seed"] == s_val)
        ]

        plotted_any = False
        for s_type, color, label in [
            ("bAP", "cyan", "bAP"),
            ("dSpike", "orange", "dSpike"),
        ]:
            data = subset[subset["spike_type"] == s_type]
            if not data.empty:
                ax.scatter(
                    data["er_orig"],
                    data["er_res"],
                    alpha=0.6,
                    c=color,
                    label=label,
                    edgecolors="k",
                    s=25,
                )
                plotted_any = True

        ax.set_xlim(x_range)
        ax.set_ylim(y_range)
        ax.set_title(f"Mean: {m}, Seed: {s_val}\n(Window: 1.0ms)")
        ax.set_xlabel("ER (Original Ca)")
        if i == 0:
            ax.set_ylabel("ER (Residual Ca)")

        if plotted_any:
            ax.legend(loc="upper right")
        ax.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()
else:
    print("df_rank_1ms が見つからないか、空です。")

# ==============================================================================
# 2. パラメータ依存性（傾き・切片）の再集計とプロット
# ==============================================================================
if "df_rank_1ms" in locals() and not df_rank_1ms.empty:
    results_1ms = []
    conditions_reg = df_rank_1ms.groupby(
        ["syn_loc_mean", "syn_loc_std", "seed"]
    ).size().index

    for m, s, seed_val in conditions_reg:
        subset = df_rank_1ms[
            (df_rank_1ms["syn_loc_mean"] == m)
            & (df_rank_1ms["syn_loc_std"] == s)
            & (df_rank_1ms["seed"] == seed_val)
        ]

        for s_type in ["bAP", "dSpike"]:
            data = subset[subset["spike_type"] == s_type]

            if len(data) > 1:
                X = data[["er_orig"]].values
                y = data["er_res"].values

                model = LinearRegression()
                model.fit(X, y)

                results_1ms.append(
                    {
                        "syn_loc_mean": m,
                        "syn_loc_std": s,
                        "seed": seed_val,
                        "spike_type": s_type,
                        "slope": model.coef_[0],
                        "intercept": model.intercept_,
                    }
                )

    res_df_1ms = pd.DataFrame(results_1ms)

    if not res_df_1ms.empty:
        summary_df_1ms = (
            res_df_1ms.groupby(["syn_loc_mean", "spike_type"])
            .agg({"slope": ["mean", "std"], "intercept": ["mean", "std"]})
            .reset_index()
        )
        summary_df_1ms.columns = [
            f"{col[0]}_{col[1]}" if col[1] else col[0]
            for col in summary_df_1ms.columns
        ]

        fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True)
        types = ["bAP", "dSpike"]
        params = ["slope", "intercept"]
        titles = ["Slope (傾き)", "Intercept (切片)"]

        for i, p_name in enumerate(params):
            for j, t_name in enumerate(types):
                ax = axes[i, j]
                plot_data = summary_df_1ms[
                    summary_df_1ms["spike_type"] == t_name
                ]

                m_vals = plot_data["syn_loc_mean"]
                mean_vals = plot_data[f"{p_name}_mean"]
                std_vals = plot_data[f"{p_name}_std"].fillna(0)

                color = "cyan" if t_name == "bAP" else "orange"

                ax.plot(
                    m_vals,
                    mean_vals,
                    marker="o",
                    linestyle="-",
                    color=color,
                    label=f"{t_name}",
                )
                ax.fill_between(
                    m_vals,
                    mean_vals - std_vals,
                    mean_vals + std_vals,
                    color=color,
                    alpha=0.15,
                )

                ax.set_title(f"{t_name} - {titles[i]} (1.0ms)")
                ax.set_ylabel(titles[i])
                ax.grid(True, linestyle="--", alpha=0.5)

                if i == 1:
                    ax.set_xlabel("syn_loc_mean")

        plt.tight_layout()
        plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==============================================================================
# 実効ランク（Original / Residual）の平均値と標準偏差をプロットするセル
# ==============================================================================
if "df_rank_1ms" in locals() and not df_rank_1ms.empty:
    # 1. mean と spike_type でグループ化し、Original と Residual の平均・標準偏差を計算
    summary_er = (
        df_rank_1ms.groupby(["syn_loc_mean", "spike_type"])
        .agg({"er_orig": ["mean", "std"], "er_res": ["mean", "std"]})
        .reset_index()
    )

    # カラム名をフラット化 (例: ('er_res', 'mean') -> 'er_res_mean')
    summary_er.columns = [
        f"{col[0]}_{col[1]}" if col[1] else col[0] for col in summary_er.columns
    ]

    # 2. 1x2 の構成でプロット（左：Originalの平均、右：Residualの平均）
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    metrics = ["er_orig", "er_res"]
    titles = [
        "Original Ca 実効ランクの平均値\n(全体ダイナミクスの複雑さ)",
        "Residual Ca 実効ランクの平均値 (PC1除去後)\n(微細構造ダイナミクスの複雑さ)",
    ]
    y_labels = ["Mean ER (Original)", "Mean ER (Residual)"]

    types = ["bAP", "dSpike"]
    palette = {"bAP": "cyan", "dSpike": "orange"}

    for idx, metric in enumerate(metrics):
        ax = axes[idx]

        for t_name in types:
            plot_data = summary_er[summary_er["spike_type"] == t_name]

            m_vals = plot_data["syn_loc_mean"]
            mean_vals = plot_data[f"{metric}_mean"]
            std_vals = plot_data[f"{metric}_std"].fillna(0)  # Seed数不足対策

            color = palette[t_name]

            # 平均線のプロット
            ax.plot(
                m_vals,
                mean_vals,
                marker="o",
                linestyle="-",
                linewidth=2,
                color=color,
                label=f"{t_name}",
            )

            # 標準偏差を影で表示
            ax.fill_between(
                m_vals,
                mean_vals - std_vals,
                mean_vals + std_vals,
                color=color,
                alpha=0.15,
            )

        ax.set_title(titles[idx], fontsize=12, pad=10)
        ax.set_xlabel("syn_loc_mean (シナプス入力の平均位置)", fontsize=11)
        ax.set_ylabel(y_labels[idx], fontsize=11)
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.legend(loc="best", fontsize=10)

    plt.tight_layout()
    plt.show()

else:
    print("df_rank_1ms が見つからないか、空です。")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==============================================================================
# 実効ランク（Original / Residual）の平均値と標準偏差をプロットするセル
# ==============================================================================
if "df_rank" in locals() and not df_rank.empty:
    # 1. mean と spike_type でグループ化し、Original と Residual の平均・標準偏差を計算
    summary_er = (
        df_rank.groupby(["syn_loc_mean", "spike_type"])
        .agg({"er_orig": ["mean", "std"], "er_res": ["mean", "std"]})
        .reset_index()
    )

    # カラム名をフラット化 (例: ('er_res', 'mean') -> 'er_res_mean')
    summary_er.columns = [
        f"{col[0]}_{col[1]}" if col[1] else col[0] for col in summary_er.columns
    ]

    # 2. 1x2 の構成でプロット（左：Originalの平均、右：Residualの平均）
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    metrics = ["er_orig", "er_res"]
    titles = [
        "Original Ca 実効ランクの平均値\n(全体ダイナミクスの複雑さ)",
        "Residual Ca 実効ランクの平均値 (PC1除去後)\n(微細構造ダイナミクスの複雑さ)",
    ]
    y_labels = ["Mean ER (Original)", "Mean ER (Residual)"]

    types = ["bAP", "dSpike"]
    palette = {"bAP": "cyan", "dSpike": "orange"}

    for idx, metric in enumerate(metrics):
        ax = axes[idx]

        for t_name in types:
            plot_data = summary_er[summary_er["spike_type"] == t_name]

            m_vals = plot_data["syn_loc_mean"]
            mean_vals = plot_data[f"{metric}_mean"]
            std_vals = plot_data[f"{metric}_std"].fillna(0)  # Seed数不足対策

            color = palette[t_name]

            # 平均線のプロット
            ax.plot(
                m_vals,
                mean_vals,
                marker="o",
                linestyle="-",
                linewidth=2,
                color=color,
                label=f"{t_name}",
            )

            # 標準偏差を影で表示
            ax.fill_between(
                m_vals,
                mean_vals - std_vals,
                mean_vals + std_vals,
                color=color,
                alpha=0.15,
            )

        ax.set_title(titles[idx], fontsize=12, pad=10)
        ax.set_xlabel("syn_loc_mean (シナプス入力の平均位置)", fontsize=11)
        ax.set_ylabel(y_labels[idx], fontsize=11)
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.legend(loc="best", fontsize=10)

    plt.tight_layout()
    plt.show()

else:
    print("df_rank が見つからないか、空です。")